In [1]:
import gc
import warnings
warnings.filterwarnings("ignore")

import torch
import chess

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

torch.manual_seed(42)
clear_memory()

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""
if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("HF login successful.")
    except Exception as e:
        print("HF login failed, proceeding without:", e)
else:
    print("No HF token provided. Proceeding without authentication.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login successful.


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-3-1b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"Loading model (dtype={dtype})...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=None if dtype is torch.float32 else dtype,
    device_map="auto",
)
# Ensure pad token id is set
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model.eval()
print("Model ready.")


Loading tokenizer...
Loading model (dtype=torch.bfloat16)...
Model ready.


In [7]:
def safe_generate(prompt: str, max_new_tokens: int = 56, temperature: float = 0.8) -> str:
    """
    Faster generation: concise outputs, strips prompt echo, handles OOM.
    """
    if not prompt.strip():
        return "[Invalid prompt]"

    device = next(model.parameters()).device
    try:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512  # keep prompt compact
        ).to(device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_p=0.85,
                temperature=temperature,   # a bit warmer for human tone
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                use_cache=True
            )

        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if prompt in text:
            text = text.split(prompt, 1)[-1]
        return text.strip() or "[Empty generation]"
    except RuntimeError as e:
        if "CUDA" in str(e):
            torch.cuda.empty_cache()
        return f"[RuntimeError: {str(e)[:80]}...]"
    except Exception as e:
        return f"[Error: {str(e)[:80]}...]"

# ---- text cleaners ----
def _strip_fences_and_labels(text: str) -> str:
    import re
    txt = re.sub(r"```.*?```", "", text, flags=re.DOTALL)
    txt = txt.replace("**Commentary:**", "").replace("Commentary:", "").strip()
    return txt

def clean_commentary(text: str, max_sentences: int = 2) -> str:
    """
    For move-by-move quips: trim to ≤ max_sentences (default 2).
    """
    import re
    txt = _strip_fences_and_labels(text)
    parts = [p.strip() for p in re.split(r'(?<=[.!?])\s+', txt) if len(p.strip()) > 6]
    if max_sentences is not None and len(parts) > max_sentences:
        parts = parts[:max_sentences]
    return " ".join(parts) if parts else "[No commentary]"

def clean_summary(text: str) -> str:
    """
    For post-game summaries: DO NOT cap sentences.
    """
    return _strip_fences_and_labels(text)

In [27]:
# Cell 7 — concept signals (engine-free) — FIXED to iterate squares safely

from collections import defaultdict

# Helper: iterate all squares occupied by a given color, robust across python-chess versions
def squares_of_color(board: chess.Board, color: chess.Color):
    for pt in (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING):
        for sq in board.pieces(pt, color):
            yield sq

def king_square(board, color):
    return board.king(color)

def count_attacks_on(board, target_sq, by_color):
    cnt = 0
    for sq in squares_of_color(board, by_color):
        if target_sq in board.attacks(sq):
            cnt += 1
    return cnt

def king_safety_signal(board, color):
    ksq = king_square(board, color)
    if ksq is None:
        return 0.0
    enemy = not color
    # ring squares around king
    ring = [ksq]
    f = chess.square_file(ksq); r = chess.square_rank(ksq)
    for df in (-1,0,1):
        for dr in (-1,0,1):
            if df==0 and dr==0: continue
            ff, rr = f+df, r+dr
            if 0 <= ff < 8 and 0 <= rr < 8:
                ring.append(chess.square(ff, rr))
    pressure = sum(count_attacks_on(board, s, enemy) for s in ring)
    # shelter: pawns in front of king (very rough)
    shelter = 0
    direction = 1 if color == chess.WHITE else -1
    for df in (-1,0,1):
        ff = f + df
        rr = r + direction
        if 0 <= ff < 8 and 0 <= rr < 8:
            sq = chess.square(ff, rr)
            p = board.piece_at(sq)
            if p and p.color == color and p.piece_type == chess.PAWN:
                shelter += 1
    return float(shelter*0.6 - pressure*0.5)

def mobility_signal(board, color):
    temp = board.copy()
    temp.turn = color
    return float(sum(1 for _ in temp.legal_moves)) * 0.03

def material_signal(board):
    val = 0.0
    for pt, w in PIECE_VALUE.items():
        val += (len(board.pieces(pt, chess.WHITE)) - len(board.pieces(pt, chess.BLACK))) * w
    return float(val)

def space_signal(board, color):
    total = 0
    for sq in squares_of_color(board, color):
        total += sum(
            1 for t in board.attacks(sq)
            if (chess.square_rank(t) >= 4 if color == chess.WHITE else chess.square_rank(t) <= 3)
        )
    return float(total) * 0.02

def passed_pawns_signal(board, color):
    score = 0.0
    enemy = not color
    for sq in board.pieces(chess.PAWN, color):
        file = chess.square_file(sq)
        rank = chess.square_rank(sq)

        # path-block (very coarse)
        blocked = False
        rng = range(rank+1,8) if color==chess.WHITE else range(rank-1,-1,-1)
        for r in rng:
            fwd = chess.square(file, r)
            p = board.piece_at(fwd)
            if p and p.color != color:
                blocked = True
                break

        # enemy pawn ahead on same/adjacent files?
        enemy_block = False
        for adj_file in (file-1, file, file+1):
            if 0 <= adj_file < 8:
                rng2 = range(rank+1,8) if color==chess.WHITE else range(rank-1,-1,-1)
                for r in rng2:
                    sq2 = chess.square(adj_file, r)
                    p = board.piece_at(sq2)
                    if p and p.color == enemy and p.piece_type == chess.PAWN:
                        enemy_block = True
                        break
            if enemy_block:
                break

        if not enemy_block:
            score += 0.5
        if not blocked:
            score += 0.2
    return float(score)

def threats_signal(board, color):
    enemy = not color
    score = 0.0
    enemy_squares = list(squares_of_color(board, enemy))
    my_squares = list(squares_of_color(board, color))
    for esq in enemy_squares:
        defenders = sum(1 for sq in enemy_squares if esq in board.attacks(sq))
        attackers = sum(1 for sq in my_squares if esq in board.attacks(sq))
        if attackers > 0 and defenders == 0:
            score += 0.4
        elif attackers > defenders:
            score += 0.2
    return float(score)

def rooks_on_open_files_signal(board, color):
    score = 0.0
    for sq in board.pieces(chess.ROOK, color):
        file = chess.square_file(sq)
        has_pawn = False
        for r in range(8):
            p = board.piece_at(chess.square(file, r))
            if p and p.piece_type == chess.PAWN:
                has_pawn = True
                break
        if not has_pawn:
            score += 0.4
    return float(score)

def queen_activity_signal(board, color):
    score = 0.0
    for sq in board.pieces(chess.QUEEN, color):
        f = chess.square_file(sq); r = chess.square_rank(sq)
        center_bonus = 0.2 if f in (3,4) and r in (3,4) else 0.0
        score += center_bonus + len(board.attacks(sq))*0.02
    return float(score)

CONCEPTS = {
    "White Material": lambda b: material_signal(b),
    "Black Material": lambda b: -material_signal(b),
    "White King Safety": lambda b: king_safety_signal(b, chess.WHITE),
    "Black King Safety": lambda b: king_safety_signal(b, chess.BLACK),
    "White Mobility": lambda b: mobility_signal(b, chess.WHITE),
    "Black Mobility": lambda b: mobility_signal(b, chess.BLACK),
    "White Space": lambda b: space_signal(b, chess.WHITE),
    "Black Space": lambda b: space_signal(b, chess.BLACK),
    "White Passed Pawns": lambda b: passed_pawns_signal(b, chess.WHITE),
    "Black Passed Pawns": lambda b: passed_pawns_signal(b, chess.BLACK),
    "White Threats": lambda b: threats_signal(b, chess.WHITE),
    "Black Threats": lambda b: threats_signal(b, chess.BLACK),
    "White Rooks on Open Files": lambda b: rooks_on_open_files_signal(b, chess.WHITE),
    "Black Rooks on Open Files": lambda b: rooks_on_open_files_signal(b, chess.BLACK),
    "White Queen Activity": lambda b: queen_activity_signal(b, chess.WHITE),
    "Black Queen Activity": lambda b: queen_activity_signal(b, chess.BLACK),
}

def concept_vector(board):
    return {name: fn(board) for name, fn in CONCEPTS.items()}

def prioritize_concepts(board_before, move, top_k=4):
    v_before = concept_vector(board_before)
    b_after = board_before.copy()
    b_after.push(move)
    v_after = concept_vector(b_after)

    deltas = []
    for k in v_before:
        d = v_after[k] - v_before[k]
        if abs(d) > 1e-6:
            deltas.append((k, d))
    deltas.sort(key=lambda x: abs(x[1]), reverse=True)
    return deltas[:top_k]

def humanize_concepts(deltas):
    if not deltas:
        return "Top concepts: (no major shifts)."
    bits = []
    for name, d in deltas:
        trend = "↑" if d > 0 else "↓"
        short = name.replace("White ", "W ").replace("Black ", "B ")
        bits.append(f"{short}{trend}")
    return "Top concepts: " + ", ".join(bits)


In [28]:
def make_comment_prompt(fen: str, san: str, verdict_line: str, last_6: str, concepts_inline: str = "") -> str:
    return (
        "You are a concise chess commentator. In ≤2 sentences, explain the *idea* of the move (plans/tactics), "
        "not just restating the SAN.\n\n"
        f"FEN: {fen}\n"
        f"Recent moves: {last_6 or 'None'}\n"
        f"Move: {san}\n"
        f"{concepts_inline}\n"
        f"Coach note: {verdict_line}\n"
        "Commentary:"
    )

def make_summary_prompt(result_str: str, reason: str, san_moves: list[str]) -> str:
    first_8 = " ".join(san_moves[:8])
    all_moves = " ".join(san_moves[-300:])
    return (
        "You are a chess analyst summarizing a completed game. "
        "Write a detailed, instructive analysis (180–250 words) for a general chess audience. "
        "Include: 1) the opening family/variation from the first moves, 2) key turning points and tactical ideas, "
        "3) examples of good and bad moves, 4) how momentum shifted, 5) the final tactic/endgame theme, 6) lessons.\n\n"
        f"Result: {result_str} ({reason}).\n"
        f"Opening moves: {first_8}\n"
        f"Full move list: {all_moves}\n\n"
        "Detailed Summary:"
    )

def print_board(board: chess.Board):
    # White at bottom; compatible with older python-chess (no 'flipped' arg)
    print(board.unicode(borders=True, invert_color=False))


In [29]:
import re

PIECE_LETTERS = set("nbrqk")
DECORATION_RE = re.compile(r"[+#?!]+")

def sanitize_san(user_input: str) -> str:
    s = user_input.strip()
    s = DECORATION_RE.sub("", s)
    return s

def normalize_san(user_input: str) -> str:
    s = sanitize_san(user_input)
    # Castling variants
    s_castle = s.replace("0-0-0", "O-O-O").replace("0-0", "O-O")
    s_castle = re.sub(r'(?i)o-o-o', "O-O-O", s_castle)
    s_castle = re.sub(r'(?i)o-o', "O-O", s_castle)
    if s_castle != s:
        return s_castle
    if s and s[0] in PIECE_LETTERS:
        s = s[0].upper() + s[1:]
    return s


In [30]:
def quick_eval_cp(board: chess.Board) -> int:
    # scale our eval_white to centipawns
    return int(eval_white(board) * 100)

def greedy_best_move_uci(board: chess.Board) -> str:
    best_score = -1e9
    best = None
    for mv in board.legal_moves:
        b2 = board.copy()
        b2.push(mv)
        sc = eval_white(b2) - eval_white(board)
        if board.is_capture(mv):
            sc += 0.15
        if board.gives_check(mv):
            sc += 0.10
        if sc > best_score:
            best_score = sc
            best = mv
    return best.uci() if best else next(iter(board.legal_moves)).uci()


In [31]:
# Cell 11 — robust loader for /mnt/data/prompts_template.txt
from pathlib import Path
import re

PROMPTS_PATH = Path("prompts_template.txt")

def load_prompt_templates(path=PROMPTS_PATH):
    if not path.exists():
        print(f"[warn] prompt file not found at {path}")
        return {}

    text = path.read_text(encoding="utf-8")
    # normalize newlines & trim trailing spaces
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [ln.rstrip() for ln in text.split("\n")]

    # We only need to detect lines like: "### Beginner ... Prompt 1"
    # Capture level and number; body runs until next "### ..." or EOF
    hdr_re = re.compile(r'^\s*###\s*(Beginner|Intermediate|Advanced)\b.*?\bPrompt\s*(\d+)\s*$', re.IGNORECASE)

    sections = {}
    current_key = None
    current_buf = []

    def flush():
        nonlocal current_key, current_buf, sections
        if current_key is not None:
            body = "\n".join(current_buf).strip()
            sections[current_key] = body
        current_key = None
        current_buf = []

    for ln in lines:
        m = hdr_re.match(ln)
        if m:
            # new section starts — flush previous
            flush()
            lvl_raw = m.group(1).strip().lower()
            try:
                lvl = {"beginner":"beginner", "intermediate":"intermediate", "advanced":"advanced"}[lvl_raw]
            except KeyError:
                # unexpected label; skip safely
                lvl = "intermediate"
            idx = int(m.group(2))
            current_key = (lvl, idx)
            current_buf = []
        else:
            # not a header — if inside a section, collect
            if current_key is not None:
                # Stop on a separator line like '---' (optional)
                if ln.strip() == '---':
                    flush()
                else:
                    current_buf.append(ln)

    # flush last block
    flush()

    if not sections:
        print("[warn] no '### <Level> ... Prompt <n>' sections found — using generic JSON prompt fallback.")
    else:
        print("Loaded prompt templates:", sorted(sections.keys()))
    return sections

PROMPTS_DB = load_prompt_templates()

Loaded prompt templates: [('advanced', 1), ('advanced', 2), ('advanced', 3), ('beginner', 1), ('beginner', 2), ('beginner', 3), ('intermediate', 1), ('intermediate', 2), ('intermediate', 3)]


In [32]:
# Cell 12 — template formatter + JSON commentary generator (with graceful fallback)

import json

# Built-in generic JSON prompt if file templates are missing or a (level,variant) key isn't found
_GENERIC_JSON_PROMPT = """You are a chess coach that returns JSON only.
Use this schema:
{
  "summary": "<1–2 sentences, plain text>",
  "plan": "<one short actionable plan for the side to move>",
  "key_moves": ["<SAN or UCI of 1–3 important candidate moves>"],
  "tags": ["<short tags like 'king safety','center','tactics'>"],
  "confidence": <0.0..1.0>
}

Context:
FEN: {fen}
EvalCP (from White POV): {eval_cp}
GreedyBest (UCI): {best_move}
RecentMoves (SAN): {last_moves}

Return strictly valid JSON, no extra text.
"""

def render_template(level: str, variant: int, fen: str, eval_cp: int, best_uci: str, last_moves_san: list[str]) -> str:
    lvl_key = (level.lower(), int(variant))
    # choose template or fallback
    if PROMPTS_DB and lvl_key in PROMPTS_DB:
        tmpl = PROMPTS_DB[lvl_key]
    elif PROMPTS_DB:
        # pick the first available template in the DB as a soft fallback
        first_key = next(iter(PROMPTS_DB.keys()))
        tmpl = PROMPTS_DB[first_key]
    else:
        tmpl = _GENERIC_JSON_PROMPT

    last = " ".join(last_moves_san[-12:]) if last_moves_san else "None"
    filled = (tmpl
              .replace("{fen}", fen)
              .replace("{eval_cp}", str(eval_cp))
              .replace("{best_move}", best_uci)
              .replace("{last_moves}", last))
    return filled

def generate_json_commentary(level: str, variant: int, board_before: chess.Board, last_moves_san: list[str]) -> dict:
    fen = board_before.fen()
    eval_cp = quick_eval_cp(board_before)
    best_uci = greedy_best_move_uci(board_before)
    prompt = render_template(level, variant, fen, eval_cp, best_uci, last_moves_san)
    raw = safe_generate(prompt, max_new_tokens=220, temperature=0.7)

    # try strict JSON
    try:
        return json.loads(raw)
    except Exception:
        # try to extract the largest {...} block
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(raw[start:end+1])
            except Exception:
                pass

    # final fallback: wrap as schema
    return {
        "summary": clean_commentary(raw, max_sentences=None),
        "key_moves": [],
        "plan": "",
        "tags": [],
        "confidence": 0.3
    }


In [33]:
# Global toggle & defaults
USE_PROMPTS = True             # set False to use free-text concept-guided commentary
PROMPT_LEVEL = "intermediate"  # 'beginner' | 'intermediate' | 'advanced'
PROMPT_VARIANT = 1             # 1 | 2 | 3

def make_move_comment(board_before: chess.Board, move: chess.Move, recent_san: list[str]):
    """
    Unified move-comment producer:
      - If USE_PROMPTS: uses your JSON templates (filled) and returns a concise 1–2 sentence line.
      - Else: returns concept-guided free-text commentary.
    """
    san_canonical = board_before.san(move)
    # Verdict + concepts
    label, reasons = classify_move(board_before, move)
    verdict_line = humanize_commentary(san_canonical, label, reasons)
    concept_deltas = prioritize_concepts(board_before, move, top_k=4)
    concepts_inline = humanize_concepts(concept_deltas)

    if USE_PROMPTS:
        data = generate_json_commentary(PROMPT_LEVEL, PROMPT_VARIANT, board_before, recent_san)
        summary = data.get("summary","").strip()
        plan = data.get("plan","").strip()
        one_liner = f"{verdict_line} {summary}"
        if plan:
            one_liner += f" ({plan})"
        return one_liner
    else:
        fen_before = board_before.fen()
        last_6 = " ".join(recent_san[-6:]) if recent_san else ""
        prompt = make_comment_prompt(fen_before, san_canonical, verdict_line, last_6, concepts_inline=concepts_inline)
        llm = clean_commentary(safe_generate(prompt, max_new_tokens=56, temperature=0.75), max_sentences=2)
        return f"{verdict_line} {llm}"


In [34]:
def run_one_game():
    print("♟️  Chess Commentary Chatbot (one game)")
    board = chess.Board()
    move_history: list[str] = []

    # show starting board (White at bottom)
    print_board(board)

    while True:
        # Terminal result -> summarize and exit
        if board.is_game_over(claim_draw=True) and move_history:
            res = board.result(claim_draw=True)     # "1-0", "0-1", "1/2-1/2"
            reason = (
                "checkmate" if board.is_checkmate() else
                "stalemate" if board.is_stalemate() else
                "insufficient material" if board.is_insufficient_material() else
                "threefold repetition" if board.can_claim_threefold_repetition() else
                "fifty-move rule" if board.can_claim_fifty_moves() else
                "game over"
            )
            print("\nGame finished:", res, f"({reason}). Generating summary...\n")
            prompt = make_summary_prompt(res, reason, move_history)
            summary = clean_summary(safe_generate(prompt, max_new_tokens=320, temperature=0.85))
            print(summary)
            move_history.clear()
            print("\nThanks for playing. Goodbye! ♟️")
            break

        user = input("\nYour move (SAN/UCI) or 'exit' to quit with summary: ").strip()
        if not user:
            continue

        # Exit early -> summarize the current *incomplete* game and exit (specify who quit)
        if user.lower() in ("exit", "quit"):
            if move_history:
                quitter = "White" if board.turn == chess.WHITE else "Black"
                print(f"\nSession ending — {quitter} quit mid-game. Generating summary...\n")
                res, reason = "*", f"{quitter} quit mid-game"
                prompt = make_summary_prompt(res, reason, move_history)
                summary = clean_summary(safe_generate(prompt, max_new_tokens=320, temperature=0.85))
                print(summary)
                move_history.clear()
            print("\nGoodbye! ♟️")
            break

        # Try SAN (strict), then UCI
        san_try = normalize_san(user)
        parsed = None
        try:
            parsed = board.parse_san(san_try)
        except Exception:
            # Try UCI
            try:
                move = chess.Move.from_uci(user.lower())
                if move not in board.legal_moves:
                    raise ValueError("Illegal UCI move")
                parsed = move
            except Exception:
                print("Invalid move. Use SAN (e.g., e4, Nf3, Bxb5+, O-O) or UCI (e2e4, g1f3).")
                continue

        # PRE-move data
        fen_before = board.fen()
        san_canonical = board.san(parsed)

        # Execute move
        board.push(parsed)
        move_history.append(san_canonical)

        # Commentary (template JSON or concept-guided free text)
        comment_line = make_move_comment(chess.Board(fen_before), parsed, move_history[:-1])

        # Print + board
        print(f"\n{san_canonical}: {comment_line}")
        print_board(board)


In [35]:
run_one_game()

♟️  Chess Commentary Chatbot (one game)
  -----------------
8 |♜|♞|♝|♛|♚|♝|♞|♜|
  -----------------
7 |♟|♟|♟|♟|♟|♟|♟|♟|
  -----------------
6 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
5 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
4 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
3 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
2 |♙|♙|♙|♙|♙|♙|♙|♙|
  -----------------
1 |♖|♘|♗|♕|♔|♗|♘|♖|
  -----------------
   a b c d e f g h



Your move (SAN/UCI) or 'exit' to quit with summary:  exit



Goodbye! ♟️


In [36]:
def summarize_moves(result_str: str, reason: str, san_moves: list[str], title: str):
    print("\n" + "="*80)
    print(f"{title} — Result: {result_str} ({reason})")
    print("="*80)
    prompt = make_summary_prompt(result_str, reason, san_moves)
    summary = clean_summary(safe_generate(prompt, max_new_tokens=320, temperature=0.85))
    print(summary)

# 1) Draw (by agreement): a calm Italian opening
draw_game = [
    "e4","e5","Nf3","Nc6","Bc4","Bc5","c3","Nf6","d3","d6","O-O","O-O",
    "Re1","a6","Bb3","Ba7","Nbd2","Be6","Nf1","Qd7","Be3","Bxe3","Nxe3","Rfe8",
    "h3","Rad8","Qe2","Ne7","Rad1","Ng6"
]

# 2) White win — Scholar’s Mate
white_win_game = ["e4","e5","Bc4","Nc6","Qh5","Nf6","Qxf7#"]

# 3) Black win — Fool’s Mate
black_win_game = ["f3","e5","g4","Qh4#"]

# 4) Someone quit mid-game — specify quitter from final turn
quit_game = ["d4","d5","c4","e6","Nc3","Nf6","Bg5","Be7","e3"]

summarize_moves("1/2-1/2", "draw agreed", draw_game, "Demo Game 1: Draw")
summarize_moves("1-0", "checkmate", white_win_game, "Demo Game 2: White Win")
summarize_moves("0-1", "checkmate", black_win_game, "Demo Game 3: Black Win")

_tmp = chess.Board()
for mv in quit_game:
    _tmp.push_san(mv)
quitter_reason = "White quit mid-game" if _tmp.turn == chess.WHITE else "Black quit mid-game"
summarize_moves("*", quitter_reason, quit_game, "Demo Game 4: Someone Quit")



Demo Game 1: Draw — Result: 1/2-1/2 (draw agreed)
The game began with a solid, positional opening, largely driven by the control of the center. The e4 e5 exchange established a clear and stable pawn structure. However, the Nc6 pawn in the opening was somewhat passive, and the game quickly shifted towards a tactical battle.

Key turning points revolved around the development of the pieces. The knight on c3, initially passive, became crucial for controlling the center. The bishop on c3 also played a pivotal role in the opening.

Tactical ideas were prevalent, particularly in the early stages. The exchange of d3 and d6 led to open lines for attack. The exchange of Rax8 led to an opening for the white queen. The final tactic involved the exchange of pieces, with the knight taking the Queen.

Momentum shifted dramatically in the mid-game. White's pressure on the center increased, while black's defense became increasingly reactive. The move e5 was a pivotal moment, creating a dynamic positi

In [ ]:
def run_scripted_game(title: str, san_moves: list[str], result_str: str, reason: str, show_boards: bool = False):
    """
    Replays a SAN move list, prints commentary per move (using current mode: templates or free-text),
    then prints a detailed post-game summary.
    """
    print("\n" + "="*100)
    print(f"{title} — Result: {result_str} ({reason})")
    print("="*100)

    board = chess.Board()
    move_history = []

    if show_boards:
        print_board(board)

    for idx, san in enumerate(san_moves, start=1):
        # Parse SAN safely (strip decorations, normalize)
        san_try = normalize_san(san)
        try:
            move = board.parse_san(san_try)
        except Exception as e:
            print(f"[Error parsing SAN '{san}' at ply {idx}: {e}]")
            break

        # Pre-move context
        fen_before = board.fen()
        canonical_san = board.san(move)

        # Execute move
        board.push(move)
        move_history.append(canonical_san)

        # Commentary (template JSON or concept-guided free text)
        comment_line = make_move_comment(chess.Board(fen_before), move, move_history[:-1])

        # Output one-liner per move
        move_no = (idx + 1) // 2
        ply_prefix = f"{move_no}..." if board.turn == chess.WHITE else f"{move_no}."
        print(f"{ply_prefix} {canonical_san}: {comment_line}")

        if show_boards:
            print_board(board)

    # --- Post-game summary ---
    if result_str == "*":
        quitter = "White" if board.turn == chess.WHITE else "Black"
        reason_full = f"{quitter} quit mid-game"
    else:
        reason_full = reason

    prompt_sum = make_summary_prompt(result_str, reason_full, move_history)
    summary = clean_summary(safe_generate(prompt_sum, max_new_tokens=320, temperature=0.85))
    print("\n— Detailed Summary —")
    print(summary)

# Reuse the four demo games defined above
run_scripted_game("Demo Game 1: Draw", draw_game, "1/2-1/2", "draw agreed", show_boards=False)
run_scripted_game("Demo Game 2: White Win", white_win_game, "1-0", "checkmate", show_boards=False)
run_scripted_game("Demo Game 3: Black Win", black_win_game, "0-1", "checkmate", show_boards=False)
run_scripted_game("Demo Game 4: Someone Quit", quit_game, "*", "someone quit mid-game", show_boards=False)



Demo Game 1: Draw — Result: 1/2-1/2 (draw agreed)
1. e4: [Good] Solid and purposeful. It improves central control. The position is a classic opening, with White having a strong pawn structure and a relatively open center. The engine's evaluation suggests White has a slight advantage, primarily due to the central pawn structure. However, the lack of tactical ideas and a slightly unbalanced position means it's difficult to determine a clear plan, and the evaluation shift is noticeable. (Develop the pieces quickly, focusing on controlling the center.  Nf3 and d3 are solid choices to prepare for further development.)
1... e5: [Inaccuracy] A bit soft. It improves central control. The position is a standard, quiet opening. The engine evaluates the position as relatively balanced, with a slight advantage for White. The king is safe, but the position is somewhat unbalanced due to the lack of active development and the potential for a tactical threat. (White should develop quickly, aiming for 

In [ ]:
def _eval_prompt(metric_name: str, board_fen: str, san: str, comment: str, engine_hint: str = ""):
    rubric = {
        "Relevance": "Focus on the move and its reasoning; avoid unrelated info.",
        "Completeness": "Cover the critical factors of the position for this move; avoid missing big factors.",
        "Clarity": "Clear, specific, non-vague; chess terms used correctly.",
        "Fluency": "Well-structured sentences; coherent and readable."
    }[metric_name]
    hint_line = f"Engine hint: {engine_hint}\n" if engine_hint else ""
    return (
        f"You are scoring a chess comment on one metric.\n"
        f"Metric: {metric_name} (1–5). Guideline: {rubric}\n"
        f"{hint_line}"
        f"Position (FEN): {board_fen}\n"
        f"Move: {san}\n"
        f"Comment: {comment}\n"
        f"Reply with a single digit 1–5. No words."
    )

def score_comment(board_before: chess.Board, move: chess.Move, comment: str, engine_hint: str = ""):
    fen = board_before.fen()
    san = board_before.san(move)
    scores = {}
    for metric in ("Relevance","Completeness","Clarity","Fluency"):
        pr = _eval_prompt(metric, fen, san, comment, engine_hint)
        raw = safe_generate(pr, max_new_tokens=2, temperature=0.0)
        try:
            s = int("".join([c for c in raw if c.isdigit()])[:1])
            s = min(5, max(1, s))
        except:
            s = 3
        scores[metric] = s
    return scores

def run_scripted_game_with_concepts(title: str, san_moves: list[str], result_str: str, reason: str,
                                    show_boards: bool = False, with_eval: bool = True):
    print("\n" + "="*100)
    print(f"{title} — Result: {result_str} ({reason})")
    print("="*100)

    board = chess.Board()
    move_history = []

    if show_boards:
        print_board(board)

    for idx, san in enumerate(san_moves, start=1):
        move = board.parse_san(normalize_san(san))
        fen_before = board.fen()
        san_canonical = board.san(move)

        # verdict + concepts
        label, reasons = classify_move(board, move)
        verdict_line = humanize_commentary(san_canonical, label, reasons)
        deltas = prioritize_concepts(board, move, top_k=4)
        concepts_inline = humanize_concepts(deltas)
        recent = " ".join(move_history[-6:]) if move_history else ""

        # push
        board.push(move)
        move_history.append(san_canonical)

        # comment via unified producer
        comment_line = make_move_comment(chess.Board(fen_before), move, move_history[:-1])

        # print one-liner with concepts
        move_no = (idx + 1)//2
        ply_prefix = f"{move_no}..." if board.turn == chess.WHITE else f"{move_no}."
        print(f"{ply_prefix} {san_canonical}: ({concepts_inline}) {comment_line}")

        if with_eval:
            scores = score_comment(chess.Board(fen_before), move, comment_line)
            print(f"    → GCC-Eval-ish: R{scores['Relevance']} C{scores['Completeness']} L{scores['Clarity']} F{scores['Fluency']}")

        if show_boards:
            print_board(board)

    if result_str == "*":
        quitter = "White" if board.turn == chess.WHITE else "Black"
        reason_full = f"{quitter} quit mid-game"
    else:
        reason_full = reason

    prompt_sum = make_summary_prompt(result_str, reason_full, move_history)
    summary = clean_summary(safe_generate(prompt_sum, max_new_tokens=320, temperature=0.85))
    print("\n— Detailed Summary —")
    print(summary)

# Example: turn on/off prompt templates or eval here before running
# USE_PROMPTS = False
# run_scripted_game_with_concepts("Demo Game 2: White Win (concepts + eval)", white_win_game, "1-0", "checkmate", show_boards=False, with_eval=True)
